<a href="https://colab.research.google.com/github/GezahegnM/Machine-Learning-and-Deep-Learning-Project-/blob/main/LLMs_Tabular_Maternal_Health_Risk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLMs as Tabular Classifiers: A Zero-Shot and Few-Shot Evaluation on Maternal Health Risk Prediction

**Research notebook — run top to bottom in Google Colab**

This notebook benchmarks Large Language Models (zero-shot and few-shot, via the Anthropic API)
against classical ML and a tabular deep learning model on the UCI **Maternal Health Risk** dataset.

**Pipeline:**
1. Data loading & cleaning (duplicate removal)
2. Exploratory Data Analysis (EDA) with visualizations
3. Preprocessing & train/test split
4. Baseline models: Logistic Regression, Random Forest, XGBoost
5. Tabular deep learning model (PyTorch MLP)
6. LLM Zero-Shot classification (row → natural language → prompt → prediction)
7. LLM Few-Shot classification (labeled examples included in the prompt)
8. Full comparison report with visual charts, confusion matrices, and a summary table

> **Before you start:** Get a free Anthropic API key at https://console.anthropic.com/ if you don't
> already have one. You'll be asked to paste it securely in Section 6 (it is never displayed or stored on disk).


## 0. Setup — Install & Import Libraries

In [ ]:
!pip install -q anthropic xgboost scikit-learn pandas matplotlib seaborn torch --upgrade


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, json, time, re, getpass

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 110

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, classification_report,
                              confusion_matrix)
from xgboost import XGBClassifier

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
print("Libraries loaded.")


## 1. Data Loading

Upload `Maternal_Health_Risk_Data_Set.csv` when prompted (Colab file picker).
If you've already placed it in the Colab filesystem or mounted Drive, edit `DATA_PATH` instead.

In [ ]:
DATA_PATH = "/content/Maternal Health Risk Data Set.csv"

try:
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded from {DATA_PATH}")
except FileNotFoundError:
    from google.colab import files
    print("Please upload Maternal_Health_Risk_Data_Set.csv")
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]
    df = pd.read_csv(DATA_PATH)

df.columns = [c.strip() for c in df.columns]
print(df.shape)
df.head()


## 2. Data Cleaning & Exploratory Data Analysis (EDA)

The raw UCI Maternal Health Risk dataset contains a large number of **exact duplicate rows**.
Prior published work on this dataset frequently reports inflated accuracy (>95%) because duplicates
leak across train/test splits. We remove duplicates before doing anything else, and report both
counts for transparency.

In [ ]:
n_before = len(df)
n_dupes = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
n_after = len(df)

print(f"Rows before dedup : {n_before}")
print(f"Duplicate rows removed : {n_dupes}")
print(f"Rows after dedup  : {n_after}")
print()
print(df['RiskLevel'].value_counts())
print()
print(df.describe())


In [ ]:
# --- Class balance ---
fig, ax = plt.subplots(figsize=(6,4))
order = df['RiskLevel'].value_counts().index
sns.countplot(data=df, x='RiskLevel', order=order, ax=ax)
ax.set_title("Class Distribution — Maternal Health Risk Level (deduplicated)")
ax.set_xlabel("Risk Level")
ax.set_ylabel("Count")
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x()+p.get_width()/2, p.get_height()),
                ha='center', va='bottom')
plt.tight_layout()
plt.savefig("fig_class_distribution.png")
plt.show()


In [ ]:
# --- Correlation heatmap ---
numeric_cols = ['Age','SystolicBP','DiastolicBP','BS','BodyTemp','HeartRate']
fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f", ax=ax)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("fig_correlation_heatmap.png")
plt.show()


In [ ]:
# --- Feature distributions by risk level ---
fig, axes = plt.subplots(2, 3, figsize=(15,8))
for ax, col in zip(axes.flat, numeric_cols):
    sns.boxplot(data=df, x='RiskLevel', y=col, order=order, ax=ax)
    ax.set_title(col)
plt.suptitle("Feature Distributions by Risk Level", y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig("fig_feature_boxplots.png")
plt.show()


## 3. Preprocessing & Train/Test Split

We keep a held-out test set fixed across *every* model (classical ML, deep learning, and LLM
prompting) so the comparison in Section 8 is apples-to-apples. LLM evaluation is run on a
sub-sample of the test set to control API cost — configurable via `LLM_TEST_SAMPLE_SIZE`.

In [ ]:
FEATURES = ['Age','SystolicBP','DiastolicBP','BS','BodyTemp','HeartRate']
TARGET = 'RiskLevel'

le = LabelEncoder()
y_all = le.fit_transform(df[TARGET])
X_all = df[FEATURES].copy()

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_all, y_all, df.index, test_size=0.25, random_state=RANDOM_STATE, stratify=y_all
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train size:", X_train.shape, "Test size:", X_test.shape)
print("Classes:", list(le.classes_))


## 4. Baseline Models — Logistic Regression, Random Forest, XGBoost

In [ ]:
results = {}   # collects metrics for every model, used in the final report

def evaluate_model(name, y_true, y_pred):
    results[name] = {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average='macro'),
        "precision_macro": precision_score(y_true, y_pred, average='macro', zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average='macro', zero_division=0),
    }
    print(f"--- {name} ---")
    print(classification_report(y_true, y_pred, target_names=le.classes_, zero_division=0))
    return confusion_matrix(y_true, y_pred)


In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_scaled, y_train)
pred_lr = lr.predict(X_test_scaled)
cm_lr = evaluate_model("Logistic Regression", y_test, pred_lr)


In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
cm_rf = evaluate_model("Random Forest", y_test, pred_rf)


In [ ]:
# XGBoost
xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                     random_state=RANDOM_STATE, eval_metric='mlogloss')
xgb.fit(X_train, y_train)
pred_xgb = xgb.predict(X_test)
cm_xgb = evaluate_model("XGBoost", y_test, pred_xgb)


## 5. Tabular Deep Learning Model — PyTorch MLP

In [ ]:
class TabularMLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(64,32)):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.BatchNorm1d(h), nn.Dropout(0.2)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)

model = TabularMLP(in_dim=X_train_scaled.shape[1], n_classes=len(le.classes_)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

EPOCHS = 100
history = []
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)
    history.append(epoch_loss / len(train_loader.dataset))
    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} - loss: {history[-1]:.4f}")

model.eval()
with torch.no_grad():
    pred_mlp = model(X_test_t.to(device)).argmax(dim=1).cpu().numpy()

cm_mlp = evaluate_model("Tabular MLP (Deep Learning)", y_test, pred_mlp)


In [ ]:
plt.figure(figsize=(6,4))
plt.plot(history)
plt.title("MLP Training Loss")
plt.xlabel("Epoch"); plt.ylabel("Cross-Entropy Loss")
plt.tight_layout()
plt.savefig("fig_mlp_loss.png")
plt.show()


## 6. LLM Zero-Shot Classification

Each test-set row is **serialized into a natural-language sentence** describing the patient's
vitals, then sent to a Claude model with a prompt that asks it to classify maternal health risk
as `low risk`, `mid risk`, or `high risk` — with **no labeled examples** given.

Paste your Anthropic API key when prompted (input is hidden, and the key is only kept in memory
for this session).

In [ ]:
ANTHROPIC_API_KEY = getpass.getpass("Enter your Anthropic API key: ")

import anthropic
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

LLM_MODEL = "claude-sonnet-4-6"          # swap for any available model string
LLM_TEST_SAMPLE_SIZE = 60                # keep small to control cost/time; raise for a fuller eval


In [ ]:
def row_to_text(row):
    return (f"Age: {int(row.Age)} years. "
            f"Systolic blood pressure: {int(row.SystolicBP)} mmHg. "
            f"Diastolic blood pressure: {int(row.DiastolicBP)} mmHg. "
            f"Blood sugar: {row.BS} mmol/L. "
            f"Body temperature: {row.BodyTemp} F. "
            f"Heart rate: {int(row.HeartRate)} bpm.")

VALID_LABELS = ["low risk", "mid risk", "high risk"]

def parse_label(text):
    text_low = text.lower()
    for lab in VALID_LABELS:
        if lab in text_low:
            return lab
    return "unknown"

def build_zero_shot_prompt(patient_text):
    return (
        "You are a clinical risk assessment assistant. Based ONLY on the vital signs below, "
        "classify the patient's maternal health risk level.\n\n"
        f"Patient vitals: {patient_text}\n\n"
        "Respond with exactly one of the following labels and nothing else: "
        "\"low risk\", \"mid risk\", or \"high risk\"."
    )

def query_llm(prompt, max_retries=3):
    for attempt in range(max_retries):
        try:
            resp = client.messages.create(
                model=LLM_MODEL,
                max_tokens=20,
                messages=[{"role": "user", "content": prompt}]
            )
            return resp.content[0].text.strip()
        except Exception as e:
            print(f"  retry {attempt+1}: {e}")
            time.sleep(2 ** attempt)
    return ""


In [ ]:
# Build the LLM evaluation subset from the SAME held-out test set used above
test_df = df.loc[idx_test].reset_index(drop=True)
test_df['true_label'] = le.inverse_transform(y_test)

llm_eval_df = test_df.sample(n=min(LLM_TEST_SAMPLE_SIZE, len(test_df)),
                              random_state=RANDOM_STATE).reset_index(drop=True)

zero_shot_preds = []
for i, row in llm_eval_df.iterrows():
    prompt = build_zero_shot_prompt(row_to_text(row))
    raw = query_llm(prompt)
    zero_shot_preds.append(parse_label(raw))
    if (i+1) % 10 == 0:
        print(f"Zero-shot: {i+1}/{len(llm_eval_df)} done")

llm_eval_df['zero_shot_pred'] = zero_shot_preds
llm_eval_df[['true_label','zero_shot_pred']].head(10)


In [ ]:
mask = llm_eval_df['zero_shot_pred'] != "unknown"
y_true_zs = llm_eval_df.loc[mask, 'true_label']
y_pred_zs = llm_eval_df.loc[mask, 'zero_shot_pred']

results['LLM Zero-Shot'] = {
    "accuracy": accuracy_score(y_true_zs, y_pred_zs),
    "f1_macro": f1_score(y_true_zs, y_pred_zs, average='macro'),
    "precision_macro": precision_score(y_true_zs, y_pred_zs, average='macro', zero_division=0),
    "recall_macro": recall_score(y_true_zs, y_pred_zs, average='macro', zero_division=0),
}
print(classification_report(y_true_zs, y_pred_zs, zero_division=0))
cm_zs = confusion_matrix(y_true_zs, y_pred_zs, labels=VALID_LABELS)


## 7. LLM Few-Shot Classification

Same procedure, but now the prompt includes a small number of **labeled example patients**
(one per class, drawn only from the training set to avoid leakage) before asking the model
to classify the new patient.

In [ ]:
train_df = df.loc[idx_train].reset_index(drop=True)
train_df['true_label'] = le.inverse_transform(y_train)

FEW_SHOT_EXAMPLES = []
for lab in VALID_LABELS:
    ex = train_df[train_df['true_label'] == lab].sample(1, random_state=RANDOM_STATE).iloc[0]
    FEW_SHOT_EXAMPLES.append((row_to_text(ex), lab))

def build_few_shot_prompt(patient_text):
    examples_block = "\n".join(
        [f"Patient vitals: {ex_text}\nRisk level: {ex_label}\n" for ex_text, ex_label in FEW_SHOT_EXAMPLES]
    )
    return (
        "You are a clinical risk assessment assistant. Below are labeled examples, followed by a "
        "new patient you must classify.\n\n"
        f"{examples_block}\n"
        f"Patient vitals: {patient_text}\n"
        "Respond with exactly one of the following labels and nothing else: "
        "\"low risk\", \"mid risk\", or \"high risk\"."
    )

few_shot_preds = []
for i, row in llm_eval_df.iterrows():
    prompt = build_few_shot_prompt(row_to_text(row))
    raw = query_llm(prompt)
    few_shot_preds.append(parse_label(raw))
    if (i+1) % 10 == 0:
        print(f"Few-shot: {i+1}/{len(llm_eval_df)} done")

llm_eval_df['few_shot_pred'] = few_shot_preds
llm_eval_df[['true_label','zero_shot_pred','few_shot_pred']].head(10)


In [ ]:
mask_fs = llm_eval_df['few_shot_pred'] != "unknown"
y_true_fs = llm_eval_df.loc[mask_fs, 'true_label']
y_pred_fs = llm_eval_df.loc[mask_fs, 'few_shot_pred']

results['LLM Few-Shot'] = {
    "accuracy": accuracy_score(y_true_fs, y_pred_fs),
    "f1_macro": f1_score(y_true_fs, y_pred_fs, average='macro'),
    "precision_macro": precision_score(y_true_fs, y_pred_fs, average='macro', zero_division=0),
    "recall_macro": recall_score(y_true_fs, y_pred_fs, average='macro', zero_division=0),
}
print(classification_report(y_true_fs, y_pred_fs, zero_division=0))
cm_fs = confusion_matrix(y_true_fs, y_pred_fs, labels=VALID_LABELS)


## 8. Full Comparison Report

All models were evaluated on the same held-out test data (LLM results on the `LLM_TEST_SAMPLE_SIZE`
sub-sample of it). Note: this is an important caveat to state explicitly in your written report.

In [ ]:
results_df = pd.DataFrame(results).T.sort_values("accuracy", ascending=False)
results_df = results_df.round(4)
results_df


In [ ]:
# --- Bar chart: accuracy & macro-F1 comparison across all models ---
fig, ax = plt.subplots(figsize=(9,5))
results_df[['accuracy','f1_macro']].plot(kind='bar', ax=ax)
ax.set_title("Model Comparison — Accuracy vs. Macro F1")
ax.set_ylabel("Score")
ax.set_xticklabels(results_df.index, rotation=30, ha='right')
ax.legend(["Accuracy", "Macro F1"])
plt.tight_layout()
plt.savefig("fig_model_comparison.png")
plt.show()


In [ ]:
# --- Confusion matrices grid ---
fig, axes = plt.subplots(2, 3, figsize=(16,9))
cm_dict = {
    "Logistic Regression": (cm_lr, le.classes_),
    "Random Forest": (cm_rf, le.classes_),
    "XGBoost": (cm_xgb, le.classes_),
    "Tabular MLP": (cm_mlp, le.classes_),
    "LLM Zero-Shot": (cm_zs, VALID_LABELS),
    "LLM Few-Shot": (cm_fs, VALID_LABELS),
}
for ax, (name, (cm, labels)) in zip(axes.flat, cm_dict.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=labels, yticklabels=labels, cbar=False)
    ax.set_title(name)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.suptitle("Confusion Matrices — All Models", y=1.02, fontsize=15)
plt.tight_layout()
plt.savefig("fig_confusion_matrices.png")
plt.show()


In [ ]:
# --- Radar chart summarizing all four metrics per model ---
from math import pi

metrics = ['accuracy','f1_macro','precision_macro','recall_macro']
categories = ['Accuracy','Macro F1','Macro Precision','Macro Recall']
n = len(categories)
angles = [i / n * 2 * pi for i in range(n)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7,7), subplot_kw=dict(polar=True))
for model_name, row in results_df.iterrows():
    values = row[metrics].tolist()
    values += values[:1]
    ax.plot(angles, values, label=model_name, linewidth=2)
    ax.fill(angles, values, alpha=0.08)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)
ax.set_title("Model Performance Radar — All Metrics", y=1.1)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=8)
plt.tight_layout()
plt.savefig("fig_radar_comparison.png")
plt.show()


## 9. Save the Report

Exports the results table and all figures for inclusion in your written research report /
paper appendix.

In [ ]:
results_df.to_csv("model_comparison_results.csv")

import zipfile
with zipfile.ZipFile("research_report_assets.zip", "w") as zf:
    for fname in ["fig_class_distribution.png","fig_correlation_heatmap.png",
                  "fig_feature_boxplots.png","fig_mlp_loss.png",
                  "fig_model_comparison.png","fig_confusion_matrices.png",
                  "fig_radar_comparison.png","model_comparison_results.csv"]:
        try:
            zf.write(fname)
        except FileNotFoundError:
            pass

print("Saved: model_comparison_results.csv and research_report_assets.zip")
print("Download research_report_assets.zip from the Colab file browser (left sidebar).")


## 10. Notes for Writing Up the Research

- **State the deduplication step explicitly** in your methodology — it materially changes reported
  accuracy vs. most prior papers on this dataset and is a legitimate methodological contribution.
- **LLM results depend heavily on prompt wording.** Report the exact prompts used (Sections 6–7) in
  your appendix, and consider running a small prompt-sensitivity ablation if time allows.
- **Sample size caveat:** LLM evaluation ran on `LLM_TEST_SAMPLE_SIZE` rows to control API cost —
  state this clearly; it is not evaluated on the full test set like the other models.
- **Expected finding:** tree-based models (XGBoost/Random Forest) will very likely outperform both
  the MLP and the LLM approaches on this small, purely numeric dataset — that is a valid and
  publishable finding, not a failure of the study. The interesting research contribution is showing
  *how close* zero/few-shot LLMs get without any training, and *how much* few-shot examples help
  over zero-shot.
